In [ ]:
# ============================================================
# Block 1: Config + Generic Utilities (REVISED, window = parameterized)
# ============================================================
import os
import glob
from collections import Counter

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import rcParams
from dask.diagnostics import ProgressBar

# ---------------- Global paths ----------------
RAW_ROOT = "/mnt/soclim0/public_data/weiji"
OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"

DATA_ROOT = os.path.join(OUT_ROOT, "data")
RANK_ROOT = os.path.join(OUT_ROOT, "rankings")
PLOT_ROOT = os.path.join(OUT_ROOT, "plots")

for p in [OUT_ROOT, DATA_ROOT, RANK_ROOT, PLOT_ROOT]:
    os.makedirs(p, exist_ok=True)

# ---------------- Experiment config ----------------
EXP_CFG = {
    "BWCN": {
        "O3_dir": os.path.join(RAW_ROOT, "BWCN", "O3"),
        "O3_pattern": "BWCN.cam.h3.*.O3.nc",
        "U_dir": os.path.join(RAW_ROOT, "BWCN", "U"),
        "U_pattern": "BWCN.cam.h3.*.U.nc",
    },
    "B2000WCN": {
        "O3_dir": os.path.join(RAW_ROOT, "B2000WCN001002", "O3"),
        "O3_pattern": "B2000WCN.sample.cam.h3.*.O3.nc",
        "U_dir": os.path.join(RAW_ROOT, "B2000WCN001002", "U"),
        "U_pattern": "B2000WCN.sample.cam.h3.*.U.nc",
    }
}

# ---------------- Plot style ----------------
rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
})

TOP_N = 5
LOW_FRAC = 0.25
TOP_COLORS = ['#1f77b4', '#d62728', '#9467bd', '#17becf', '#2ca02c']


# ============================================================
# Generic file / data helpers
# ============================================================
def parse_file_year(path):
    """Extract physical file-year token from filename."""
    return int(os.path.basename(path).split(".")[-3])


def open_daily_files(exp, data_dir, pattern, chunks=None):
    """
    Generic reader:
      - BWCN: normal open_mfdataset
      - B2000WCN: split into Run001 (1-103) + Run002 (105-210), then shift Run002 by +103 years
    """
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"[{exp}] No files found: {os.path.join(data_dir, pattern)}")

    if exp == "B2000WCN":
        files_001 = [f for f in files if 1 <= parse_file_year(f) <= 103]
        files_002 = [f for f in files if 105 <= parse_file_year(f) <= 210]

        if len(files_001) == 0 or len(files_002) == 0:
            raise RuntimeError(f"[{exp}] Run001 / Run002 split failed.")

        print(f"[{exp}] Loading Run001: {len(files_001)} files")
        ds1 = xr.open_mfdataset(files_001, combine="by_coords", chunks=chunks)

        print(f"[{exp}] Loading Run002: {len(files_002)} files (shift +103 years)")
        ds2 = xr.open_mfdataset(files_002, combine="by_coords", chunks=chunks)

        new_times = [t.replace(year=t.year + 103) for t in ds2.time.values]
        ds2 = ds2.assign_coords(time=new_times)

        ds = xr.concat([ds1, ds2], dim="time").sortby("time")
    else:
        print(f"[{exp}] Loading {len(files)} files")
        ds = xr.open_mfdataset(files, combine="by_coords", chunks=chunks).sortby("time")

    return ds


def find_lev_name(da):
    for name in ("lev", "plev", "lev_p", "level"):
        if name in da.dims or name in da.coords:
            return name
    raise ValueError(f"Cannot find vertical coordinate in dims={da.dims}")


def normalize_lev_to_pa(da):
    """Rename vertical coord to 'lev' and convert hPa -> Pa if needed."""
    lev_name = find_lev_name(da)
    lev_vals = da[lev_name].values
    lev_pa = lev_vals * 100.0 if float(np.nanmax(lev_vals)) <= 2000.0 else lev_vals

    if lev_name != "lev":
        da = da.rename({lev_name: "lev"})
    da = da.assign_coords(lev=("lev", lev_pa))
    da["lev"].attrs["units"] = "Pa"
    return da


def extract_at_lat(ds, var, lat0, lon_reduce="mean"):
    """
    Generic extractor:
      - optionally zonal mean over lon
      - select nearest latitude
      - normalize vertical coord to Pa
    Returns DataArray(time, lev)
    """
    da = ds[var]

    if "lon" in da.dims and lon_reduce == "mean":
        da = da.mean("lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension")

    actual_lat = float(da["lat"].sel(lat=lat0, method="nearest").values)
    da = da.sel(lat=lat0, method="nearest")
    da = normalize_lev_to_pa(da)

    print(f"[extract_at_lat] Requested {lat0}°, actual grid latitude = {actual_lat:.2f}°")
    return da


def select_level_da(da_2d, target_hpa):
    """
    Select nearest vertical level from DataArray(time, lev).
    Returns:
      da_1d(time), actual_hpa
    """
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(da_2d["lev"].values - target_pa)))
    actual_hpa = float(da_2d["lev"].values[idx] / 100.0)
    return da_2d.isel(lev=idx), actual_hpa


def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    """
    Hybrid-coordinate O3 column integration (DU) over [p_top_hpa, p_bot_hpa].
    """
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    factor = Na / (g * M_air * 2.687e20)

    P0 = ds_sub["P0"]
    PS = ds_sub["PS"]

    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS

    p_i = (
        P_interface.isel(ilev=slice(0, -1))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )
    p_ip1 = (
        P_interface.isel(ilev=slice(1, None))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    return (ds_sub["O3"] * overlap * factor).sum(dim="lev")


# ============================================================
# Window helpers (parameterized)
# Default intended use now: JUL(Y-1) -> JUN(Y), i.e. start_month=7
# ============================================================
def get_window_month_order(start_month=7):
    """
    Return ordered 12-month sequence starting from start_month.
    Example:
      start_month=7 -> [7,8,9,10,11,12,1,2,3,4,5,6]
    """
    return list(range(start_month, 13)) + list(range(1, start_month))


def get_window_month_labels(start_month=7):
    month_names = {
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
        7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
    }
    return [month_names[m] for m in get_window_month_order(start_month)]


def build_shifted_line(da_1d, year, start_month=7):
    """
    Build one cross-year line:
      start_month(Y-1) -> month(start_month-1)(Y)

    Example:
      start_month=7 -> Jul(Y-1) ... Jun(Y)

    Returns:
      values (1D np.ndarray) or None
      meta dict or None
    """
    da_1d = da_1d.sortby("time")

    yy = da_1d.time.dt.year.values.astype(int)
    mm = da_1d.time.dt.month.values.astype(int)

    # Previous year: months >= start_month
    # Current year : months < start_month
    mask = ((yy == (year - 1)) & (mm >= start_month)) | ((yy == year) & (mm < start_month))
    idx = np.where(mask)[0]

    if idx.size == 0:
        return None, None

    seg = da_1d.isel(time=idx).sortby("time")
    seg_months = seg.time.dt.month.values.astype(int)

    month_order = get_window_month_order(start_month)
    month_counts = [int(np.sum(seg_months == m)) for m in month_order]

    # All 12 months must be present
    if np.any(np.array(month_counts) == 0):
        return None, None

    # Check month order (month blocks must appear in correct sequence)
    unique_months_in_order = []
    for m in seg_months:
        if len(unique_months_in_order) == 0 or int(m) != unique_months_in_order[-1]:
            unique_months_in_order.append(int(m))

    if unique_months_in_order != month_order:
        return None, None

    values = seg.values.astype(float)
    meta = {
        "year": int(year),
        "n_days": int(len(values)),
        "month_counts": month_counts,
    }
    return values, meta


def scan_valid_windows(series_map, start_month=7):
    """
    Scan all years in all experiments and determine the dominant valid window length.
    This automatically supports 365_day / noleap / 360_day model calendars.
    """
    scan_rows = []

    for exp, da in series_map.items():
        years = np.unique(da.time.dt.year.values.astype(int))
        for y in years:
            _, meta = build_shifted_line(da, int(y), start_month=start_month)
            if meta is None:
                continue

            scan_rows.append({
                "exp": exp,
                "year": int(y),
                "n_days": int(meta["n_days"]),
                "month_counts": meta["month_counts"],
            })

    if len(scan_rows) == 0:
        raise RuntimeError("No valid cross-year windows found.")

    scan_df = pd.DataFrame(scan_rows)

    counts = Counter(scan_df["n_days"].tolist())
    target_n_days = counts.most_common(1)[0][0]

    ref_row = scan_df[scan_df["n_days"] == target_n_days].iloc[0]
    ref_month_counts = list(ref_row["month_counts"])

    return scan_df, target_n_days, ref_month_counts


def make_xticks_from_month_counts(month_counts):
    """
    Build month-start xticks dynamically from actual month lengths.
    Example:
      noleap Jul-Jun -> [0,31,62,...]
      360_day Jul-Jun -> [0,30,60,...]
    """
    xticks = [0]
    cum = 0
    for c in month_counts[:-1]:
        cum += int(c)
        xticks.append(cum)
    return xticks


def materialize_lines_from_rows(series_map, rows_df, target_n_days, start_month=7):
    """
    Build pooled lines from exp-year rows, keeping only lines whose n_days == target_n_days.
    Returns:
      lines (n, target_n_days)
      kept_rows_df
    """
    lines = []
    kept = []

    for _, row in rows_df.iterrows():
        exp = str(row["exp"])
        year = int(row["year"])

        if exp not in series_map:
            continue

        vals, meta = build_shifted_line(series_map[exp], year, start_month=start_month)
        if vals is None:
            continue

        if meta["n_days"] != target_n_days:
            continue

        lines.append(vals)
        kept.append({"exp": exp, "year": year})

    if len(lines) == 0:
        return np.empty((0, target_n_days), dtype=float), pd.DataFrame(columns=["exp", "year"])

    return np.vstack(lines), pd.DataFrame(kept)


def build_all_year_rows(series_map):
    rows = []
    for exp, da in series_map.items():
        years = np.unique(da.time.dt.year.values.astype(int))
        for y in years:
            rows.append({"exp": exp, "year": int(y)})
    return pd.DataFrame(rows)


# ============================================================
# Generic pooled plotting function (window-parameterized)
# ============================================================
def plot_pooled_timeseries_style(
    series_map,
    ranking_df,
    out_png,
    ylabel,
    title,
    top_n=5,
    low_frac=0.25,
    ylim=None,
    add_zero_line=False,
    save_pdf=True,
    start_month=7,
):
    """
    Generic pooled line plot for any 1D daily series:
      - pooled all-year climatology ±1σ
      - pooled low-O3 composite ±1σ
      - pooled top-N special years from ranking_df
    Window is parameterized by start_month.

    Default:
      start_month=7 -> JUL(Y-1) to JUN(Y)
    """
    month_labels = get_window_month_labels(start_month)

    # ---------- infer valid window length ----------
    _, target_n_days, ref_month_counts = scan_valid_windows(series_map, start_month=start_month)
    xticks = make_xticks_from_month_counts(ref_month_counts)

    # ---------- all-year pooled ----------
    all_rows = build_all_year_rows(series_map)
    all_lines, _ = materialize_lines_from_rows(
        series_map, all_rows, target_n_days, start_month=start_month
    )
    if len(all_lines) == 0:
        raise RuntimeError("No valid all-year windows found for plotting.")

    all_mean = np.nanmean(all_lines, axis=0)
    all_std = np.nanstd(all_lines, axis=0)

    # ---------- pooled low-O3 ----------
    n_low = max(1, int(np.floor(low_frac * len(ranking_df))))
    low_rows = ranking_df.sort_values("rank").head(n_low)[["exp", "year"]]

    low_lines, _ = materialize_lines_from_rows(
        series_map, low_rows, target_n_days, start_month=start_month
    )
    if len(low_lines) == 0:
        raise RuntimeError("No valid low-O3 windows found for plotting.")

    low_mean = np.nanmean(low_lines, axis=0)
    low_std = np.nanstd(low_lines, axis=0)

    # ---------- pooled top-N special years ----------
    ranked_rows = ranking_df.sort_values("rank")[["exp", "year"]]
    ranked_lines_valid, ranked_kept = materialize_lines_from_rows(
        series_map, ranked_rows, target_n_days, start_month=start_month
    )
    if len(ranked_lines_valid) == 0:
        raise RuntimeError("No valid ranked windows found for plotting.")

    top_lines = ranked_lines_valid[:top_n]
    top_kept = ranked_kept.iloc[:top_n].reset_index(drop=True)

    print("\nSelected special years for plotting:")
    print(top_kept.to_string(index=False))

    # ---------- plot ----------
    fig, ax = plt.subplots(figsize=(6.8, 4.2), constrained_layout=True)
    x = np.arange(target_n_days)

    # all-year climatology
    ax.fill_between(
        x, all_mean - all_std, all_mean + all_std,
        color="0.85", alpha=0.95, zorder=0
    )
    ax.plot(x, all_mean, color="black", ls="--", lw=1.8, zorder=2)

    # low-O3 composite
    ax.fill_between(
        x, low_mean - low_std, low_mean + low_std,
        facecolor="none", edgecolor="0.5", hatch="///", zorder=1
    )
    ax.plot(x, low_mean, color="black", ls="-", lw=2.0, zorder=3)

    # special-year lines
    year_handles = []
    for i in range(len(top_lines)):
        exp = str(top_kept.loc[i, "exp"])
        year = int(top_kept.loc[i, "year"])
        label = f"{exp}-{year:04d}"
        color = TOP_COLORS[i % len(TOP_COLORS)]

        ax.plot(x, top_lines[i], color=color, lw=1.6, zorder=10)
        year_handles.append(Line2D([0], [0], color=color, lw=1.6, label=label))

    # axes
    ax.set_xlim(0, target_n_days - 1)
    ax.set_xticks(xticks)
    ax.set_xticklabels(month_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    if add_zero_line:
        ax.axhline(0, color="black", lw=0.8, alpha=0.5)

    if ylim is not None:
        ax.set_ylim(*ylim)

    # legend
    handles = year_handles + [
        Line2D([0], [0], color="black", lw=1.8, ls="--", label="Pooled all-year"),
        Patch(facecolor="0.85", label="Pooled all-year ±1σ"),
        Line2D([0], [0], color="black", lw=2.0, ls="-", label="Pooled low-O3"),
        Patch(facecolor="none", edgecolor="0.5", hatch="///", label="Pooled low-O3 ±1σ"),
    ]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")

    if save_pdf:
        out_pdf = out_png.replace(".png", ".pdf")
        plt.savefig(out_pdf, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved: {out_png}")

In [ ]:
# ============================================================
# Block 2: Prepare O3 cache (30-70 hPa) + pooled ranking
# ============================================================

O3_PTOP = 30
O3_PBOT = 70
O3_TAG = f"{O3_PTOP}_{O3_PBOT}hPa"

def prepare_o3_cache(exp, p_top_hpa=30, p_bot_hpa=70, force=False):
    """
    Heavy step (only once):
      - compute 60-90N polar-cap O3 column in [p_top_hpa, p_bot_hpa]
      - save to NetCDF cache
    """
    out_dir = os.path.join(DATA_ROOT, exp)
    os.makedirs(out_dir, exist_ok=True)
    out_nc = os.path.join(out_dir, f"O3_pc_{exp}_{p_top_hpa}_{p_bot_hpa}hPa.nc")

    if os.path.exists(out_nc) and not force:
        print(f"[{exp}] O3 cache exists, skip: {out_nc}")
        return out_nc

    cfg = EXP_CFG[exp]
    ds = open_daily_files(
        exp,
        cfg["O3_dir"],
        cfg["O3_pattern"],
        chunks={"time": 365}
    )

    print(f"[{exp}] Computing O3 polar-cap column {p_top_hpa}-{p_bot_hpa} hPa ...")
    ds_cap = ds.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(ds_cap["lat"]))

    o3_op = calc_parc_o3_hybrid(ds_cap, p_top_hpa, p_bot_hpa).weighted(weights).mean(dim=["lat", "lon"])

    with ProgressBar():
        da = o3_op.compute()

    da.name = "O3_pc_60_90N"
    da.attrs["units"] = "DU"
    da.to_netcdf(out_nc)

    year_min = int(da.time.dt.year.min().values)
    year_max = int(da.time.dt.year.max().values)
    print(f"[{exp}] O3 cache saved: {out_nc}")
    print(f"[{exp}] year range: {year_min} - {year_max}, ntime = {da.sizes['time']}")

    ds.close()
    return out_nc


def compute_spring_o3_min_5d(da_o3):
    """
    5-day running mean, then spring (Mar-Apr) yearly minimum.
    Keep only years with >=58 valid spring days.
    """
    da_5d = da_o3.rolling(time=5, center=True, min_periods=5).mean()
    spring = da_5d.where(da_5d.time.dt.month.isin([3, 4]), drop=True)

    day_counts = spring.groupby("time.year").count()
    valid_years = day_counts.year.where(day_counts >= 58, drop=True)

    spring_min = spring.groupby("time.year").min().sel(year=valid_years)
    return spring_min


def build_pooled_o3_ranking(p_top_hpa=30, p_bot_hpa=70, top_n=5, low_frac=0.25):
    """
    Lightweight step:
      - load per-exp O3 caches
      - compute pooled O3 ranking (ascending = lower ozone = more extreme)
      - save CSVs
    """
    rows = []

    for exp in ["BWCN", "B2000WCN"]:
        o3_nc = os.path.join(DATA_ROOT, exp, f"O3_pc_{exp}_{p_top_hpa}_{p_bot_hpa}hPa.nc")
        if not os.path.exists(o3_nc):
            raise FileNotFoundError(f"Missing O3 cache: {o3_nc}")

        da = xr.open_dataarray(o3_nc)
        spring_min = compute_spring_o3_min_5d(da)

        df = pd.DataFrame({
            "exp": exp,
            "year": spring_min["year"].values.astype(int),
            "o3_min": spring_min.values.astype(float),
        })
        rows.append(df)

        print(f"[{exp}] valid spring years = {len(df)}")

    rank_df = pd.concat(rows, ignore_index=True)
    rank_df = rank_df.sort_values("o3_min", ascending=True).reset_index(drop=True)
    rank_df["rank"] = np.arange(1, len(rank_df) + 1)

    n_low = max(1, int(np.floor(low_frac * len(rank_df))))
    top_df = rank_df.head(top_n).copy()
    low_df = rank_df.head(n_low).copy()

    full_csv = os.path.join(RANK_ROOT, f"pooled_O3_rank_{p_top_hpa}_{p_bot_hpa}hPa.csv")
    top_csv  = os.path.join(RANK_ROOT, f"pooled_O3_top{top_n}_{p_top_hpa}_{p_bot_hpa}hPa.csv")
    low_csv  = os.path.join(RANK_ROOT, f"pooled_O3_low25pct_{p_top_hpa}_{p_bot_hpa}hPa.csv")

    rank_df.to_csv(full_csv, index=False)
    top_df.to_csv(top_csv, index=False)
    low_df.to_csv(low_csv, index=False)

    print("\n[Pooled O3 Ranking] Top entries:")
    print(top_df.to_string(index=False))
    print(f"\nSaved full ranking : {full_csv}")
    print(f"Saved top{top_n}     : {top_csv}")
    print(f"Saved low25%     : {low_csv}")

    return rank_df


# ---------------- Run Block 2 ----------------
for exp_name in ["BWCN", "B2000WCN"]:
    prepare_o3_cache(exp_name, p_top_hpa=O3_PTOP, p_bot_hpa=O3_PBOT, force=False)

rank_df = build_pooled_o3_ranking(p_top_hpa=O3_PTOP, p_bot_hpa=O3_PBOT, top_n=TOP_N, low_frac=LOW_FRAC)

In [ ]:
# ============================================================
# Block 3: Prepare U cache (60N zonal mean)
# ============================================================

def prepare_u60n_cache(exp, force=False):
    """
    Heavy step (only once):
      - read raw U daily files
      - compute zonal-mean U at 60N -> DataArray(time, lev)
      - save to NetCDF cache
    """
    out_dir = os.path.join(DATA_ROOT, exp)
    os.makedirs(out_dir, exist_ok=True)
    out_nc = os.path.join(out_dir, f"U_60N_{exp}.nc")

    if os.path.exists(out_nc) and not force:
        print(f"[{exp}] U cache exists, skip: {out_nc}")
        return out_nc

    cfg = EXP_CFG[exp]
    ds = open_daily_files(
        exp,
        cfg["U_dir"],
        cfg["U_pattern"],
        chunks={"time": 365, "lat": 48}
    )

    print(f"[{exp}] Computing U at 60N (zonal mean) ...")
    u_op = extract_at_lat(ds, var="U", lat0=60.0, lon_reduce="mean")

    with ProgressBar():
        u_da = u_op.compute()

    u_da = u_da.transpose("time", "lev")
    u_da.name = "U_60N"
    u_da.attrs["units"] = "m/s"
    u_da["lev"].attrs["units"] = "Pa"

    u_da.to_netcdf(out_nc)

    year_min = int(u_da.time.dt.year.min().values)
    year_max = int(u_da.time.dt.year.max().values)
    lev_min = float(u_da.lev.min().values / 100.0)
    lev_max = float(u_da.lev.max().values / 100.0)

    print(f"[{exp}] U cache saved: {out_nc}")
    print(f"[{exp}] year range: {year_min} - {year_max}, ntime = {u_da.sizes['time']}")
    print(f"[{exp}] level range: {lev_min:.2f} - {lev_max:.2f} hPa")

    ds.close()
    return out_nc


# ---------------- Run Block 3 ----------------
for exp_name in ["BWCN", "B2000WCN"]:
    prepare_u60n_cache(exp_name, force=False)

In [ ]:
# ============================================================
# Block 4: Plot pooled U line charts (REVISED, JUL(Y-1) -> JUN(Y))
# ============================================================

# 你以后只需要改这里：
TARGET_LEVELS_HPA = [1.0, 10.0, 50.0, 100.0]   # 可改成 [10.0] 或 [10.0, 30.0, 70.0]
SAVE_PDF = True

# 新窗口：JUL(Y-1) -> JUN(Y)
WINDOW_START_MONTH = 7   # 7 = July

# 可选固定 y 轴；没有就自动
YLIM_MAP = {
    1.0:   (-40, 100),
    10.0:  (-20,  70),
    50.0:  (-10,  50),
    100.0: ( -5,  35),
}


def load_u_series_map_at_level(target_hpa):
    """
    Load cached U(time, lev), select one target level, return:
      series_map = {"BWCN": da(time), "B2000WCN": da(time)}
    """
    series_map = {}

    for exp in ["BWCN", "B2000WCN"]:
        u_nc = os.path.join(DATA_ROOT, exp, f"U_60N_{exp}.nc")
        if not os.path.exists(u_nc):
            raise FileNotFoundError(f"Missing U cache: {u_nc}")

        da_2d = xr.open_dataarray(u_nc)
        da_1d, actual_hpa = select_level_da(da_2d, target_hpa)

        print(f"[{exp}] target = {target_hpa:.1f} hPa, actual = {actual_hpa:.2f} hPa")
        series_map[exp] = da_1d

    return series_map


def plot_u_pooled_levels(target_levels_hpa):
    """
    Plot pooled U(60N) line charts using:
      - pooled O3 ranking (30-70 hPa, 5-day running mean spring minima)
      - pooled all-year / low-O3 climatology
      - window = JUL(Y-1) -> JUN(Y)
    """
    rank_csv = os.path.join(RANK_ROOT, f"pooled_O3_rank_{O3_PTOP}_{O3_PBOT}hPa.csv")
    if not os.path.exists(rank_csv):
        raise FileNotFoundError(f"Missing pooled ranking CSV: {rank_csv}")

    ranking_df = pd.read_csv(rank_csv)

    plot_dir = os.path.join(PLOT_ROOT, "U_pooled_60N")
    os.makedirs(plot_dir, exist_ok=True)

    for target_hpa in target_levels_hpa:
        print("\n" + "=" * 70)
        print(f"Plotting pooled U at target level: {target_hpa:.1f} hPa")

        series_map = load_u_series_map_at_level(target_hpa)
        ylim = YLIM_MAP.get(float(target_hpa), None)

        out_png = os.path.join(
            plot_dir,
            f"POOLED_U_60N_{int(round(target_hpa))}hPa_JulJun_top{TOP_N}_vs_climatology.png"
        )

        plot_pooled_timeseries_style(
            series_map=series_map,
            ranking_df=ranking_df,
            out_png=out_png,
            ylabel=f"U at 60°N, ~{int(round(target_hpa))} hPa (m/s)",
            title=(
                f"Pooled 60°N Zonal Mean Zonal Wind | ~{int(round(target_hpa))} hPa\n"
                f"JUL(Y-1) to JUN(Y) | Special years ranked by pooled O$_3$ minima "
                f"({O3_PTOP}–{O3_PBOT} hPa)"
            ),
            top_n=TOP_N,
            low_frac=LOW_FRAC,
            ylim=ylim,
            add_zero_line=True,
            save_pdf=SAVE_PDF,
            start_month=WINDOW_START_MONTH,
        )


# ---------------- Run Block 4 ----------------
plot_u_pooled_levels(TARGET_LEVELS_HPA)